<a href="https://colab.research.google.com/github/evildead23151/3D-Projects/blob/main/Project_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
# ============================================================
# 0. Install dependencies (local, cached, safe)
# ============================================================
!pip install open3d plotly

# ============================================================
# 1. Imports
# ============================================================
import numpy as np
import open3d as o3d
import plotly.graph_objects as go

# ============================================================
# 2. Physically-correct synthetic LiDAR / Sonar generator
# ============================================================
"""
We simulate a rotating range sensor:

- Multiple vertical beams (rings)
- Horizontal rotation
- Ground plane
- Vertical wall
- Occlusion
- Range-based intensity
- Structured noise

This matches LiDAR & sonar geometry.
"""

np.random.seed(0)

NUM_VERTICAL_BEAMS = 32
NUM_HORIZONTAL_STEPS = 720
MAX_RANGE = 50.0

vertical_angles = np.linspace(-25, 5, NUM_VERTICAL_BEAMS) * np.pi / 180
horizontal_angles = np.linspace(-np.pi, np.pi, NUM_HORIZONTAL_STEPS)

points = []
intensity = []

for v in vertical_angles:
    for h in horizontal_angles:
        # Ray direction
        dx = np.cos(v) * np.cos(h)
        dy = np.cos(v) * np.sin(h)
        dz = np.sin(v)

        # Ignore invalid rays
        if abs(dz) < 1e-6:
            continue

        # --- Scene ---
        # Ground plane: z = -2
        t_ground = (-2.0) / dz if dz < 0 else np.inf

        # Vertical wall: x = 25
        t_wall = 25.0 / dx if dx > 0 else np.inf

        t = min(t_ground, t_wall)

        if t < 1.0 or t > MAX_RANGE:
            continue

        x = t * dx + np.random.normal(0, 0.02)
        y = t * dy + np.random.normal(0, 0.02)
        z = t * dz + np.random.normal(0, 0.02)

        points.append([x, y, z])
        intensity.append(np.clip(1.0 / t + np.random.rand() * 0.05, 0, 1))

points = np.asarray(points)
intensity = np.asarray(intensity)

print(f"Generated {points.shape[0]} sensor points")

# ============================================================
# 3. Open3D Point Cloud
# ============================================================
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points)

# ============================================================
# 4. Headless-safe visualization (Plotly)
# ============================================================
fig = go.Figure(
    data=[
        go.Scatter3d(
            x=points[:, 0],
            y=points[:, 1],
            z=points[:, 2],
            mode="markers",
            marker=dict(
                size=1,
                color=intensity,
                colorscale="Viridis",
                opacity=0.9
            )
        )
    ]
)

fig.update_layout(
    title="Synthetic LiDAR / Sonar-style Point Cloud (100% Offline)",
    scene=dict(
        xaxis_title="X (forward)",
        yaxis_title="Y (left)",
        zaxis_title="Z (up)",
        aspectmode="data"
    ),
    margin=dict(l=0, r=0, t=40, b=0)
)

fig.show()

# ============================================================
# 5. Save for all future steps
# ============================================================
o3d.io.write_point_cloud("/content/synthetic_sensor_pointcloud.ply", pcd)
print("Saved: /content/synthetic_sensor_pointcloud.ply")


Generated 19200 sensor points


Saved: /content/synthetic_sensor_pointcloud.ply
